# 3.1 轻量化架构设计

## 什么是轻量化架构？

从架构层面设计更适合端侧部署的模型，使其在参数量更少的情况下达到与大模型可比的性能。核心思想：通过精心设计训练数据和训练策略，使小参数量模型达到甚至超越更大模型的性能。

## 为什么用轻量化架构？

1. **根本性效率提升**：轻量级架构从设计之初就考虑效率，比后期压缩（量化/剪枝）更根本地降低计算量。
2. **端侧友好**：专为移动端/嵌入式设备设计，在有限算力下仍能运行。
3. **实时推理**：满足实时应用（如目标检测、语音识别）的延迟要求。

## 轻量化架构的核心策略

- **深度可分离卷积**：将标准卷积分解为深度卷积+逐点卷积，计算量降至$1/C_{out}$
- **组卷积**：将通道分组独立卷积，计算量降至$1/g$
- **通道重排**：在组卷积间混洗通道，实现跨组信息流动
- **知识蒸馏训练**：用大模型的输出指导小模型训练
- **数据增强**：用更多/更好的训练数据弥补模型容量的不足

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch version: {torch.__version__}")

---
### 深度可分离卷积（Depthwise Separable Convolution）

#### 什么是深度可分离卷积？

将标准卷积分解为两步：
1. **深度卷积（Depthwise Conv）**：每个输入通道独立卷积，不跨通道混合
2. **逐点卷积（Pointwise Conv）**：$1 \times 1$卷积实现跨通道信息融合

#### 为什么深度可分离卷积有效？

标准卷积计算量：$C_{in} \times C_{out} \times K^2 \times H \times W$

深度可分离卷积计算量：$C_{in} \times K^2 \times H \times W + C_{in} \times C_{out} \times H \times W$

计算量比值：
$$\frac{C_{in} K^2 + C_{in} C_{out}}{C_{in} C_{out} K^2} = \frac{1}{C_{out}} + \frac{1}{K^2}$$

当$C_{out}=256, K=3$时，计算量仅为标准卷积的约$1/9$。

In [ ]:
class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_channels, in_channels, kernel_size,
            stride=stride, padding=padding, groups=in_channels, bias=False
        )
        self.pointwise = nn.Conv2d(in_channels, out_channels, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.bn1(x)
        x = F.silu(x)
        x = self.pointwise(x)
        x = self.bn2(x)
        x = F.silu(x)
        return x

class StandardConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels, kernel_size,
            stride=stride, padding=padding, bias=False
        )
        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = F.silu(x)
        return x

in_ch, out_ch, k = 64, 128, 3
std_conv = StandardConv(in_ch, out_ch, k)
ds_conv = DepthwiseSeparableConv(in_ch, out_ch, k)

std_flops = in_ch * out_ch * k * k
ds_flops = in_ch * k * k + in_ch * out_ch

std_params = sum(p.numel() for p in std_conv.parameters())
ds_params = sum(p.numel() for p in ds_conv.parameters())

x = torch.randn(1, in_ch, 32, 32)
with torch.no_grad():
    for _ in range(10):
        std_conv(x)
    start = time.perf_counter()
    for _ in range(50):
        std_conv(x)
    std_time = (time.perf_counter() - start) / 50 * 1000

    for _ in range(10):
        ds_conv(x)
    start = time.perf_counter()
    for _ in range(50):
        ds_conv(x)
    ds_time = (time.perf_counter() - start) / 50 * 1000

print(f"=== 深度可分离卷积 vs 标准卷积 ===")
print(f"配置: in={in_ch}, out={out_ch}, kernel={k}x{k}")
print(f"\n{'指标':<20} {'标准卷积':<15} {'深度可分离':<15} {'比值':<10}")
print("-" * 60)
print(f"{'FLOPs(理论)':<20} {std_flops:<15,} {ds_flops:<15,} {ds_flops/std_flops:.3f}")
print(f"{'参数量':<20} {std_params:<15,} {ds_params:<15,} {ds_params/std_params:.3f}")
print(f"{'推理延迟(ms)':<20} {std_time:<15.3f} {ds_time:<15.3f} {ds_time/std_time:.3f}")
print(f"\n深度可分离卷积将计算量降至约1/{std_flops//ds_flops}，是MobileNet系列的核心构建块")

---
### MobileNetV2：倒残差结构（Inverted Residual）

#### 什么是倒残差结构？

传统残差：宽→窄→宽（bottleneck）

倒残差：窄→宽→窄（expand → depthwise → project）

1. **扩展（Expand）**：$1 \times 1$卷积将通道扩展$t$倍
2. **深度卷积（Depthwise）**：在扩展后的高维空间做空间滤波
3. **投影（Project）**：$1 \times 1$卷积投影回低维（无激活函数，线性瓶颈）

#### 为什么倒残差更好？

1. **线性瓶颈**：投影层不加激活函数，避免低维空间的信息损失
2. **短路连接**：在窄通道间建立残差连接，梯度流动更顺畅
3. **扩展比**：在高维空间做非线性变换，表达能力更强

In [ ]:
class InvertedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expand_ratio=6):
        super().__init__()
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio
        layers = []
        if expand_ratio != 1:
            layers.extend([
                nn.Conv2d(in_channels, hidden_dim, 1, bias=False),
                nn.BatchNorm2d(hidden_dim),
                nn.SiLU(inplace=True),
            ])
        layers.extend([
            nn.Conv2d(hidden_dim, hidden_dim, 3, stride=stride, padding=1,
                      groups=hidden_dim, bias=False),
            nn.BatchNorm2d(hidden_dim),
            nn.SiLU(inplace=True),
            nn.Conv2d(hidden_dim, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        ])
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_residual:
            return x + self.conv(x)
        return self.conv(x)

class MobileNetV2(nn.Module):
    def __init__(self, num_classes=1000, width_mult=1.0):
        super().__init__()
        config = [
            [1, 16, 1, 1],
            [6, 24, 2, 2],
            [6, 32, 3, 2],
            [6, 64, 4, 2],
            [6, 96, 3, 1],
            [6, 160, 3, 2],
            [6, 320, 1, 1],
        ]
        input_channels = int(32 * width_mult)
        self.stem = nn.Sequential(
            nn.Conv2d(3, input_channels, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(input_channels),
            nn.SiLU(inplace=True),
        )
        self.blocks = nn.Sequential()
        for t, c, n, s in config:
            output_channels = int(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                self.blocks.append(
                    InvertedResidualBlock(input_channels, output_channels, stride, t)
                )
                input_channels = output_channels
        last_channels = int(1280 * width_mult) if width_mult > 1.0 else 1280
        self.head = nn.Sequential(
            nn.Conv2d(input_channels, last_channels, 1, bias=False),
            nn.BatchNorm2d(last_channels),
            nn.SiLU(inplace=True),
        )
        self.classifier = nn.Linear(last_channels, num_classes)
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.pool(x).flatten(1)
        x = self.classifier(x)
        return x

mobilenet = MobileNetV2(num_classes=1000, width_mult=1.0)
mobilenet_small = MobileNetV2(num_classes=1000, width_mult=0.5)

full_params = sum(p.numel() for p in mobilenet.parameters())
small_params = sum(p.numel() for p in mobilenet_small.parameters())

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = mobilenet(x)
    out_small = mobilenet_small(x)

print(f"=== MobileNetV2 架构 ===")
print(f"MobileNetV2 (1.0x): {full_params/1e6:.2f}M参数, 输出{out.shape}")
print(f"MobileNetV2 (0.5x): {small_params/1e6:.2f}M参数, 输出{out_small.shape}")
print(f"\n宽度乘数0.5x将参数量降至{small_params/full_params*100:.1f}%")
print(f"\nMobileNetV2关键设计:")
print(f"  1. 倒残差+线性瓶颈: 在高维做非线性，低维做线性投影")
print(f"  2. 深度可分离卷积: 计算量远低于标准卷积")
print(f"  3. 宽度乘数/分辨率乘数: 灵活调节模型大小适配不同设备")

---
### 通道重排（Channel Shuffle）与 ShuffleNet

#### 什么是通道重排？

组卷积虽然降低计算量，但不同组之间没有信息交换。通道重排通过重新排列通道顺序，使下一层组卷积能接收来自不同组的输入。

#### 为什么通道重排有效？

1. **跨组信息流动**：解决组卷积的信息隔离问题
2. **无参数开销**：仅是内存重排，不增加任何参数或计算量
3. **与组卷积互补**：组卷积降计算量，通道重排保信息流动

#### 数学原理

设输入$X \in \mathbb{R}^{C \times H \times W}$，分组数为$g$：
1. Reshape: $X \to \mathbb{R}^{g \times (C/g) \times H \times W}$
2. Transpose: 交换前两维 $\to \mathbb{R}^{(C/g) \times g \times H \times W}$
3. Flatten: $\to \mathbb{R}^{C \times H \times W}$

这样每个组的输入都包含了上一轮所有组的信息。

In [ ]:
def channel_shuffle(x, groups):
    B, C, H, W = x.shape
    channels_per_group = C // groups
    x = x.view(B, groups, channels_per_group, H, W)
    x = x.transpose(1, 2).contiguous()
    x = x.view(B, -1, H, W)
    return x

class ShuffleNetUnit(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, groups=2):
        super().__init__()
        self.stride = stride
        self.groups = groups
        mid_channels = out_channels // 4

        self.bottleneck = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 1, groups=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.SiLU(inplace=True),
        )
        self.depthwise = nn.Sequential(
            nn.Conv2d(mid_channels, mid_channels, 3, stride=stride, padding=1,
                      groups=mid_channels, bias=False),
            nn.BatchNorm2d(mid_channels),
        )
        self.expand = nn.Sequential(
            nn.Conv2d(mid_channels, out_channels, 1, groups=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.use_residual = (stride == 1 and in_channels == out_channels)
        if stride > 1:
            self.avgpool = nn.AvgPool2d(3, stride=2, padding=1)

    def forward(self, x):
        residual = x
        out = self.bottleneck(x)
        out = channel_shuffle(out, self.groups)
        out = self.depthwise(out)
        out = self.expand(out)
        if self.use_residual:
            return out + residual
        return torch.cat([out, self.avgpool(residual)], dim=1)

class ShuffleNetV2(nn.Module):
    def __init__(self, num_classes=1000, stages_out_channels=[24, 116, 232, 464, 1024], groups=2):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, stages_out_channels[0], 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(stages_out_channels[0]),
            nn.SiLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        input_channels = stages_out_channels[0]
        self.stages = nn.Sequential()
        for i, out_ch in enumerate(stages_out_channels[1:-1]):
            stage = nn.Sequential()
            for j in range(4):
                stride = 2 if j == 0 else 1
                stage.append(ShuffleNetUnit(input_channels, out_ch, stride, groups))
                input_channels = out_ch
            self.stages.append(stage)
        self.head = nn.Sequential(
            nn.Conv2d(input_channels, stages_out_channels[-1], 1, bias=False),
            nn.BatchNorm2d(stages_out_channels[-1]),
            nn.SiLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(stages_out_channels[-1], num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.stages(x)
        x = self.head(x)
        x = self.pool(x).flatten(1)
        x = self.classifier(x)
        return x

shufflenet = ShuffleNetV2(num_classes=1000)
shuffle_params = sum(p.numel() for p in shufflenet.parameters())

x = torch.randn(1, 3, 224, 224)
with torch.no_grad():
    out = shufflenet(x)

x_test = torch.randn(1, 64, 14, 14)
x_shuffled = channel_shuffle(x_test, groups=2)

print(f"=== ShuffleNetV2 架构 ===")
print(f"参数量: {shuffle_params/1e6:.2f}M")
print(f"输出形状: {out.shape}")
print(f"\n通道重排验证:")
print(f"  输入形状: {x_test.shape}")
print(f"  重排后形状: {x_shuffled.shape}")
print(f"  值是否改变: {not torch.equal(x_test, x_shuffled)}")
print(f"  元素总数不变: {x_test.numel() == x_shuffled.numel()}")
print(f"\nShuffleNetV2关键设计:")
print(f"  1. 通道重排: 无参数的跨组信息交换")
print(f"  2. 通道分割(非组卷积): 直接将通道一分为二，更高效")
print(f"  3. 输入通道=C输出通道: 确保MAC(内存访问成本)最优")

---
### 小参数量模型（SLM）设计

研究表明，通过高质量训练数据和知识蒸馏，小参数量模型（1B-3B）可以达到甚至超越更大模型的性能。代表：Phi系列、Gemma-2B、MiniCPM。

关键设计原则：
1. **深而窄**：更多层+更窄的隐藏维度
2. **嵌入共享**：输入输出嵌入共享权重
3. **旋转位置编码（RoPE）**：比学习位置编码更高效

In [ ]:
class RotaryPositionEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=512, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        t = torch.arange(max_seq_len).float()
        freqs = torch.outer(t, inv_freq)
        self.register_buffer('cos_cache', freqs.cos())
        self.register_buffer('sin_cache', freqs.sin())

    def forward(self, x, seq_len=None):
        if seq_len is None:
            seq_len = x.shape[2]
        cos = self.cos_cache[:seq_len].unsqueeze(0).unsqueeze(0)
        sin = self.sin_cache[:seq_len].unsqueeze(0).unsqueeze(0)
        x1, x2 = x[..., ::2], x[..., 1::2]
        rotated = torch.stack([-x2, x1], dim=-1).reshape_as(x)
        return x * cos + rotated * sin

class LightweightAttention(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads=None):
        super().__init__()
        n_kv_heads = n_kv_heads or n_heads
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.n_rep = n_heads // n_kv_heads
        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryPositionEmbedding(self.head_dim)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.n_kv_heads, self.head_dim).transpose(1, 2)
        q = self.rope(q, N)
        k = self.rope(k, N)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)
        attn = (q @ k.transpose(-2, -1)) * (self.head_dim ** -0.5)
        attn = attn.softmax(dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.out_proj(out)

class LightweightBlock(nn.Module):
    def __init__(self, dim, n_heads, n_kv_heads=None, ffn_mult=4):
        super().__init__()
        self.attn = LightweightAttention(dim, n_heads, n_kv_heads)
        self.ffn = nn.Sequential(
            nn.Linear(dim, dim * ffn_mult, bias=False),
            nn.SiLU(),
            nn.Linear(dim * ffn_mult, dim, bias=False),
        )
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

class SmallLanguageModel(nn.Module):
    def __init__(self, vocab_size=32000, dim=512, n_layers=12, n_heads=8,
                 n_kv_heads=2, ffn_mult=4, tied_embeddings=True):
        super().__init__()
        self.tied_embeddings = tied_embeddings
        self.emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            LightweightBlock(dim, n_heads, n_kv_heads, ffn_mult)
            for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(dim)
        if not tied_embeddings:
            self.lm_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, input_ids):
        h = self.emb(input_ids)
        for layer in self.layers:
            h = layer(h)
        h = self.ln_f(h)
        if self.tied_embeddings:
            logits = F.linear(h, self.emb.weight)
        else:
            logits = self.lm_head(h)
        return logits

configs = {
    'SLM-125M': {'dim': 512, 'n_layers': 12, 'n_heads': 8, 'n_kv_heads': 2},
    'SLM-350M': {'dim': 768, 'n_layers': 18, 'n_heads': 12, 'n_kv_heads': 4},
    'SLM-1B':   {'dim': 1024, 'n_layers': 24, 'n_heads': 16, 'n_kv_heads': 4},
}

print(f"=== SLM架构参数对比 ===")
print(f"{'模型':<12} {'dim':<6} {'layers':<8} {'heads':<7} {'kv_heads':<10} {'参数量':<12} {'嵌入共享节省':<12}")
print("-" * 67)

for name, cfg in configs.items():
    model = SmallLanguageModel(vocab_size=32000, **cfg, tied_embeddings=True)
    params = sum(p.numel() for p in model.parameters())
    emb_params = 32000 * cfg['dim']
    print(f"{name:<12} {cfg['dim']:<6} {cfg['n_layers']:<8} {cfg['n_heads']:<7} {cfg['n_kv_heads']:<10} {params/1e6:.1f}M      {emb_params/1e6:.1f}M")

print(f"\n关键设计:")
print(f"1. GQA(n_kv_heads < n_heads): 减少KV Cache，加速推理")
print(f"2. 嵌入共享: 输入输出共享emb权重，节省{32000*512/1e6:.0f}M参数")
print(f"3. RoPE: 无需学习位置编码，外推性更好")
print(f"4. SiLU激活: 比GELU计算更简单，效果相当")

### 深度与宽度的最优权衡

#### 什么是深度-宽度权衡？

研究表明，在相同参数预算下，更深更窄的网络比浅更宽的网络更适合语言建模任务。这一发现来自对小型模型缩放规律的系统研究。

#### 为什么更深更窄更好？

1. **更强的表示能力**：每增加一层相当于增加一次非线性变换，组合表示能力指数增长
2. **参数效率**：深层窄网络的参数利用效率更高，相同FLOPs下性能更好
3. **实践验证**：Phi系列、TinyLlama等小模型都采用更深更窄的设计

#### 数学原理

参数预算$P$下，深度$d$和宽度$w$的关系：
$$P \approx 12 \cdot d \cdot w^2 \quad \text{(Transformer)}$$

最优深度-宽度比：$d^* / w^* \approx 40 \sim 80$

计算量（FLOPs）与深度的关系：
$$\text{FLOPs} \propto d \cdot w^2 \cdot L_{\text{seq}}$$

在固定FLOPs预算下，增加$d$减少$w$（保持$P$不变）通常能获得更好的性能。

In [ ]:
def create_transformer(vocab_size, dim, n_layers, n_heads=4):
    model = SmallLanguageModel(
        vocab_size=vocab_size, dim=dim, n_layers=n_layers,
        n_heads=n_heads, n_kv_heads=2, tied_embeddings=True
    )
    return model

vocab_size = 5000
target_params = 2_000_000

configs = [
    ('浅宽', 256, 4),
    ('中等', 192, 8),
    ('深窄', 128, 16),
    ('极深窄', 96, 24),
]

print(f"=== 深度 vs 宽度权衡 (目标~{target_params/1e6:.0f}M参数) ===")
print(f"{'配置':<10} {'dim':<6} {'layers':<8} {'参数量':<12} {'每层参数':<12}")
print("-" * 48)

for name, dim, n_layers in configs:
    model = create_transformer(vocab_size, dim, n_layers)
    params = sum(p.numel() for p in model.parameters())
    layer_params = params / n_layers
    print(f"{name:<10} {dim:<6} {n_layers:<8} {params/1e6:.2f}M      {layer_params/1e3:.1f}K")

x = torch.randint(0, vocab_size, (2, 32))
results = {}
for name, dim, n_layers in configs:
    model = create_transformer(vocab_size, dim, n_layers)
    params = sum(p.numel() for p in model.parameters())
    with torch.no_grad():
        start = time.perf_counter()
        for _ in range(10):
            out = model(x)
        elapsed = time.perf_counter() - start
    results[name] = {'params': params, 'time': elapsed, 'dim': dim, 'layers': n_layers}

print(f"\n推理速度对比 (10次前向):")
for name, r in results.items():
    print(f"  {name}: {r['time']:.4f}s ({r['dim']}x{r['layers']})")

print(f"\n结论: 深窄模型在相同参数量下通常表现更好，但推理延迟更高")
print(f"端侧部署需在精度和延迟间权衡，通常选择中等深度")

---
### 知识蒸馏（Knowledge Distillation）

#### 什么是知识蒸馏？

用大模型（教师模型）的输出分布指导小模型（学生模型）训练，使小模型学到教师模型的"暗知识"（dark knowledge）——即类别间的相似性信息。

#### 为什么知识蒸馏对轻量化架构重要？

1. **弥补容量不足**：小模型参数有限，蒸馏可传递教师模型的类别间关系知识
2. **训练数据增强**：教师模型的软标签比硬标签包含更多信息
3. **产业实践**：Phi系列、DistilBERT、MiniCPM均使用蒸馏训练

#### 数学原理

蒸馏损失函数：
$$L = \alpha \cdot L_{\text{CE}}(y, y_s) + (1-\alpha) \cdot T^2 \cdot L_{\text{KL}}(\sigma(z_t/T), \sigma(z_s/T))$$

其中：
- $L_{\text{CE}}$：学生模型与真实标签的交叉熵
- $L_{\text{KL}}$：教师与学生软标签的KL散度
- $T$：温度参数（$T>1$使分布更平滑，暴露类别间关系）
- $\alpha$：损失权重（通常0.5-0.7）
- $z_t, z_s$：教师/学生的logits

In [ ]:
class DistillationLoss(nn.Module):
    def __init__(self, temperature=4.0, alpha=0.7):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()
        self.kl_loss = nn.KLDivLoss(reduction='batchmean')

    def forward(self, student_logits, teacher_logits, labels):
        hard_loss = self.ce_loss(student_logits, labels)
        soft_student = F.log_softmax(student_logits / self.temperature, dim=-1)
        soft_teacher = F.softmax(teacher_logits / self.temperature, dim=-1)
        soft_loss = self.kl_loss(soft_student, soft_teacher) * (self.temperature ** 2)
        return self.alpha * hard_loss + (1 - self.alpha) * soft_loss

class TeacherModel(nn.Module):
    def __init__(self, vocab_size=5000, dim=256, n_layers=6, n_heads=8):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim, nhead=n_heads, dim_feedforward=dim*4,
                activation='gelu', batch_first=True
            ) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    def forward(self, input_ids):
        h = self.emb(input_ids)
        for layer in self.layers:
            h = layer(h)
        h = self.ln_f(h)
        return self.lm_head(h)

class StudentModel(nn.Module):
    def __init__(self, vocab_size=5000, dim=64, n_layers=3, n_heads=4):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
        self.layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim, nhead=n_heads, dim_feedforward=dim*2,
                activation='gelu', batch_first=True
            ) for _ in range(n_layers)
        ])
        self.ln_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    def forward(self, input_ids):
        h = self.emb(input_ids)
        for layer in self.layers:
            h = layer(h)
        h = self.ln_f(h)
        return self.lm_head(h)

vocab_size = 5000
teacher = TeacherModel(vocab_size, dim=256, n_layers=6, n_heads=8)
student = StudentModel(vocab_size, dim=64, n_layers=3, n_heads=4)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

distill_loss = DistillationLoss(temperature=4.0, alpha=0.7)
ce_only_loss = nn.CrossEntropyLoss()

batch_size, seq_len = 4, 32
input_ids = torch.randint(0, vocab_size, (batch_size, seq_len))
labels = torch.randint(0, vocab_size, (batch_size, seq_len))

teacher.eval()
with torch.no_grad():
    teacher_logits = teacher(input_ids)

student_logits = student(input_ids)
labels_flat = labels.view(-1)
student_logits_flat = student_logits.view(-1, vocab_size)
teacher_logits_flat = teacher_logits.view(-1, vocab_size)

loss_distill = distill_loss(student_logits_flat, teacher_logits_flat, labels_flat)
loss_ce_only = ce_only_loss(student_logits_flat, labels_flat)

print(f"=== 知识蒸馏训练 ===")
print(f"教师模型: {teacher_params/1e6:.2f}M参数 (dim=256, 6层)")
print(f"学生模型: {student_params/1e6:.2f}M参数 (dim=64, 3层)")
print(f"压缩比: {teacher_params/student_params:.1f}x")
print(f"\n损失对比:")
print(f"  纯交叉熵损失: {loss_ce_only.item():.4f}")
print(f"  蒸馏损失(α=0.7, T=4.0): {loss_distill.item():.4f}")
print(f"\n温度T的作用:")
for T in [1, 2, 4, 8, 16]:
    soft_t = F.softmax(teacher_logits_flat[0] / T, dim=-1)
    entropy = -(soft_t * soft_t.log()).sum().item()
    top5_prob = soft_t.topk(5)[0]
    print(f"  T={T:<3} 熵={entropy:.2f} Top5概率={[f'{p:.3f}' for p in top5_prob.tolist()]}")

print(f"\n蒸馏训练流程:")
print(f"  1. 教师模型前向 -> teacher_logits (无需梯度)")
print(f"  2. 学生模型前向 -> student_logits")
print(f"  3. 计算蒸馏损失 = α*CE(student, labels) + (1-α)*T²*KL(soft_teacher, soft_student)")
print(f"  4. 反向传播仅更新学生模型参数")

---
### 知识蒸馏训练实战

以下展示一个完整的蒸馏训练循环，演示学生模型如何在教师模型指导下学习。

In [ ]:
torch.manual_seed(42)

teacher = TeacherModel(vocab_size, dim=256, n_layers=6, n_heads=8)
student = StudentModel(vocab_size, dim=64, n_layers=3, n_heads=4)
teacher.eval()

distill_criterion = DistillationLoss(temperature=4.0, alpha=0.7)
ce_criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-3)

n_steps = 30
distill_losses = []
ce_losses = []

for step in range(n_steps):
    input_ids = torch.randint(0, vocab_size, (4, 32))
    labels = torch.randint(0, vocab_size, (4 * 32,))

    with torch.no_grad():
        teacher_logits = teacher(input_ids).view(-1, vocab_size)

    student_logits = student(input_ids).view(-1, vocab_size)

    loss_distill = distill_criterion(student_logits, teacher_logits, labels)
    loss_ce = ce_criterion(student_logits, labels)

    optimizer.zero_grad()
    loss_distill.backward()
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    optimizer.step()

    distill_losses.append(loss_distill.item())
    ce_losses.append(loss_ce.item())

print(f"=== 蒸馏训练过程 ({n_steps}步) ===")
print(f"{'步骤':<8} {'蒸馏损失':<12} {'CE损失':<12}")
print("-" * 32)
for i in [0, 5, 10, 15, 20, 25, 29]:
    if i < len(distill_losses):
        print(f"{i+1:<8} {distill_losses[i]:<12.4f} {ce_losses[i]:<12.4f}")

print(f"\n损失变化:")
print(f"  初始蒸馏损失: {distill_losses[0]:.4f}")
print(f"  最终蒸馏损失: {distill_losses[-1]:.4f}")
print(f"  下降比例: {(1 - distill_losses[-1]/distill_losses[0])*100:.1f}%")
print(f"\n产业级蒸馏技巧:")
print(f"  1. 渐进蒸馏: 大教师→中教师→小模型，逐步压缩")
print(f"  2. 特征蒸馏: 不仅匹配logits，还匹配中间层特征")
print(f"  3. 数据增强蒸馏: 教师对增强数据生成软标签")
print(f"  4. 在线蒸馏: 多个模型同时训练互为教师")

---
### 轻量化架构端侧部署实战

#### 端侧部署的关键考量

1. **模型大小**：设备存储有限，模型需压缩到可接受范围
2. **推理延迟**：实时应用要求低延迟（<30ms/帧）
3. **内存占用**：运行时内存（含激活值）不能超出设备限制
4. **算子兼容性**：端侧推理框架（NCNN/TFLite/ONNX Runtime）支持的算子有限

#### 部署优化流程

```
训练模型 → 导出ONNX → 量化(INT8/INT4) → 算子融合 → 端侧推理
```

#### 关键优化技术

- **算子融合**：Conv+BN+ReLU融合为单个算子，减少内存访问
- **INT8量化**：权重和激活均量化为INT8，4x参数压缩
- **动态量化**：仅量化权重，激活保持FP16，精度损失更小
- **图优化**：常量折叠、死代码消除、公共子表达式消除

In [ ]:
def fuse_conv_bn(conv, bn):
    fused_conv = nn.Conv2d(
        conv.in_channels, conv.out_channels, conv.kernel_size,
        stride=conv.stride, padding=conv.padding, groups=conv.groups, bias=True
    )
    with torch.no_grad():
        w_conv = conv.weight.clone()
        bn_mean = bn.running_mean
        bn_var = bn.running_var
        bn_gamma = bn.weight
        bn_beta = bn.bias
        bn_eps = bn.eps
        std = torch.sqrt(bn_var + bn_eps)
        gamma_over_std = bn_gamma / std
        fused_conv.weight.copy_(w_conv * gamma_over_std.view(-1, 1, 1, 1))
        if conv.bias is not None:
            fused_conv.bias.copy_((conv.bias - bn_mean) / std * bn_gamma + bn_beta)
        else:
            fused_conv.bias.copy_(bn_beta - bn_mean * gamma_over_std)
    return fused_conv

class FusedMobileNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, expand_ratio=6):
        super().__init__()
        self.use_residual = (stride == 1 and in_channels == out_channels)
        hidden_dim = in_channels * expand_ratio
        if expand_ratio != 1:
            self.expand_conv = nn.Conv2d(in_channels, hidden_dim, 1, bias=True)
            self.expand_act = nn.SiLU(inplace=True)
        self.depthwise_conv = nn.Conv2d(hidden_dim, hidden_dim, 3, stride=stride, padding=1,
                                         groups=hidden_dim, bias=True)
        self.depthwise_act = nn.SiLU(inplace=True)
        self.project_conv = nn.Conv2d(hidden_dim, out_channels, 1, bias=True)

    def forward(self, x):
        residual = x
        if hasattr(self, 'expand_conv'):
            x = self.expand_act(self.expand_conv(x))
        x = self.depthwise_act(self.depthwise_conv(x))
        x = self.project_conv(x)
        if self.use_residual:
            x = x + residual
        return x

def profile_model(model, input_size=(1, 3, 224, 224), n_runs=20):
    model.eval()
    x = torch.randn(*input_size)
    params = sum(p.numel() for p in model.parameters())
    fp16_size = params * 2 / 1024 / 1024
    int8_size = params * 1 / 1024 / 1024

    with torch.no_grad():
        for _ in range(5):
            model(x)
        start = time.perf_counter()
        for _ in range(n_runs):
            model(x)
        avg_time = (time.perf_counter() - start) / n_runs * 1000

    return {'params': params, 'fp16_mb': fp16_size, 'int8_mb': int8_size, 'latency_ms': avg_time}

mobilenet_v2 = MobileNetV2(num_classes=1000, width_mult=1.0)
mobilenet_v2_small = MobileNetV2(num_classes=1000, width_mult=0.35)

profile_full = profile_model(mobilenet_v2)
profile_small = profile_model(mobilenet_v2_small)

print(f"=== 端侧部署模型对比 ===")
print(f"{'指标':<20} {'MobileNetV2-1.0':<18} {'MobileNetV2-0.35':<18}")
print("-" * 56)
print(f"{'参数量':<20} {profile_full['params']/1e6:<18.2f} {profile_small['params']/1e6:<18.2f}")
print(f"{'FP16大小(MB)':<20} {profile_full['fp16_mb']:<18.2f} {profile_small['fp16_mb']:<18.2f}")
print(f"{'INT8大小(MB)':<20} {profile_full['int8_mb']:<18.2f} {profile_small['int8_mb']:<18.2f}")
print(f"{'推理延迟(ms)':<20} {profile_full['latency_ms']:<18.2f} {profile_small['latency_ms']:<18.2f}")

conv = nn.Conv2d(64, 128, 3, padding=1, bias=False)
bn = nn.BatchNorm2d(128)
conv.eval()
bn.eval()
fused = fuse_conv_bn(conv, bn)

x = torch.randn(1, 64, 14, 14)
with torch.no_grad():
    orig_out = bn(conv(x))
    fused_out = fused(x)
    max_diff = (orig_out - fused_out).abs().max().item()

print(f"\n=== Conv+BN融合验证 ===")
print(f"原始输出与融合输出最大差异: {max_diff:.2e}")
print(f"融合后参数: Conv权重{fused.weight.shape} + 偏置{fused.bias.shape}")
print(f"\n端侧部署优化清单:")
print(f"  1. Conv+BN+ReLU融合: 减少算子数量和内存访问")
print(f"  2. INT8量化: 模型大小降至1/4，推理速度提升2-4x")
print(f"  3. 宽度乘数调节: 灵活适配不同算力设备")
print(f"  4. ONNX导出 + 算子优化: 确保端侧框架兼容")
print(f"  5. 知识蒸馏: 压缩后用蒸馏恢复精度")

---
### 轻量化架构总结

| 技术 | 计算量降低 | 精度影响 | 适用场景 |
|------|-----------|---------|----------|
| 深度可分离卷积 | ~8-9x | 小 | CNN视觉模型 |
| 组卷积+通道重排 | ~2-4x | 小 | CNN视觉模型 |
| 倒残差结构 | ~4-8x | 小 | 移动端视觉 |
| GQA注意力 | KV Cache降~4x | 极小 | 语言模型 |
| 嵌入共享 | 参数省16-50M | 无 | 语言模型 |
| 知识蒸馏 | 无直接降低 | 恢复精度 | 所有模型 |
| 深窄设计 | 同参数更优 | 正面 | 语言模型 |

**端侧部署最佳实践**：
1. 从架构设计入手（深度可分离/GQA），比后期压缩更根本
2. 用宽度乘数/分辨率乘数灵活适配设备
3. 量化+蒸馏联合使用：先蒸馏压缩，再量化加速
4. 算子融合+图优化减少运行时开销
5. 导出ONNX后用NCNN/TFLite等框架部署